In [ ]:
import numpy as np
import csv
import matplotlib.pyplot as plt
import torch
import glob
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import os
import torch.nn as nn
from PIL import Image
import h5py
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
%autosave 30

In [2]:
import random
import numpy as np
import torch


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 尽量让 CUDA 计算确定
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
g = torch.Generator()
g.manual_seed(42)

set_seed(42)

In [3]:
import h5py
import numpy as np
import os
from PIL import Image
import torch
from torch.utils.data import Dataset


class SVHNMultiDigitDataset(Dataset):
    """
    SVHN original format (multi-digit).

    Each sample returns:
        img   : transformed PIL image (RGB)
        label : LongTensor of shape (MAX_DIGITS + 1,)
                label[0]          = num_digits (1-5)
                label[1..n]       = digit values (0-9)
                label[n+1..end]   = PADDING (10)
    """

    MAX_DIGITS = 5
    PADDING = 10

    def __init__(self, img_dir, mat_path, transform=None):
        """
        :param img_dir:  path to image folder, e.g. .../train/
        :param mat_path: path to digitStruct.mat, e.g. .../train_digitStruct.mat
        :param transform: torchvision transforms applied to the cropped image
        """
        self.img_dir = img_dir
        self.transform = transform
        self.metadata = self._parse_mat(mat_path)

    def _parse_mat(self, mat_path):
        metadata = []

        with h5py.File(mat_path, "r") as f:
            ds = f["digitStruct"]
            n = ds["name"].shape[0]

            for i in range(n):
                # name is stored as (n_chars, 1) uint16 — flatten so chr() gets scalars
                name = "".join(
                    chr(int(c)) for c in f[ds["name"][i, 0]][:].flatten()
                )

                bbox = f[ds["bbox"][i, 0]]

                # bind bbox via default arg to avoid closure-over-loop-variable bug
                def _get(key, bbox=bbox):
                    arr = bbox[key]
                    if arr.shape[0] == 1:
                        return [int(arr[0, 0])]
                    return [int(f[arr[j, 0]][0, 0]) for j in range(arr.shape[0])]

                labels  = _get("label")
                lefts   = _get("left")
                tops    = _get("top")
                widths  = _get("width")
                heights = _get("height")

                # SVHN uses 10 to represent digit 0
                labels = [0 if l == 10 else l for l in labels]

                metadata.append({
                    "name":    name,
                    "labels":  labels,
                    "left":    lefts,
                    "top":     tops,
                    "width":   widths,
                    "height":  heights,
                })

        return metadata

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        meta = self.metadata[idx]

        img = Image.open(os.path.join(self.img_dir, meta["name"])).convert("RGB")
        W, H = img.size

        # union bounding box of all digits + 30% padding
        left   = min(meta["left"])
        top    = min(meta["top"])
        right  = max(l + w for l, w in zip(meta["left"],  meta["width"]))
        bottom = max(t + h for t, h in zip(meta["top"],   meta["height"]))

        pad_w = 0.3 * (right - left)
        pad_h = 0.3 * (bottom - top)

        left   = max(0, int(left   - pad_w))
        top    = max(0, int(top    - pad_h))
        right  = min(W, int(right  + pad_w))
        bottom = min(H, int(bottom + pad_h))

        img = img.crop((left, top, right, bottom))

        if self.transform:
            img = self.transform(img)

        # label: [num_digits, d1, d2, d3, d4, d5]
        digits = meta["labels"][: self.MAX_DIGITS]
        label = np.full(self.MAX_DIGITS + 1, self.PADDING, dtype=np.int64)
        label[0] = len(digits)
        label[1 : len(digits) + 1] = digits

        return img, torch.tensor(label, dtype=torch.long)


In [4]:
from torchvision import transforms
from torch.utils.data import DataLoader

data_root = "../../../../../data/SVHN"

train_transform = transforms.Compose(
    [
        transforms.Resize((64, 128)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.Resize((64, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ]
)

# images are nested: train/train/, test/test/ (extracted into same-named subfolder)
train_dataset = SVHNMultiDigitDataset(
    img_dir=os.path.join(data_root, "train", "train"),
    mat_path=os.path.join(data_root, "train_digitStruct.mat"),
    transform=train_transform,
)

test_dataset = SVHNMultiDigitDataset(
    img_dir=os.path.join(data_root, "test", "test"),
    mat_path=os.path.join(data_root, "test_digitStruct.mat"),
    transform=test_transform,
)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    drop_last=True,
    generator=g,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

print(f"train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"test:  {len(test_dataset)} samples, {len(test_loader)} batches")

# sanity check
# imgs, labels = next(iter(train_loader))
# print(f"imgs shape:   {imgs.shape}")    # [B, 3, 64, 64]
# print(f"labels shape: {labels.shape}")  # [B, 6]  — [num_digits, d1..d5]
# print(f"sample label: {labels[0]}")


train: 33402 samples, 521 batches
test:  13068 samples, 205 batches


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class WideBasicBlock(nn.Module):
    """
    WideResNet basic block for CIFAR-10.

    Structure:
        BN -> ReLU -> Conv
        BN -> ReLU -> Dropout -> Conv
        + shortcut

    This is closer to pre-activation ResNet / WideResNet style.
    """

    def __init__(self, in_channels, out_channels, stride=1, drop_rate=0.0):
        super().__init__()

        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=drop_rate) if drop_rate > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                padding=0,
                bias=False,
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)

        out = self.bn1(x)
        out = self.relu1(out)
        out = self.conv1(out)

        out = self.bn2(out)
        out = self.relu2(out)
        out = self.dropout(out)
        out = self.conv2(out)

        out = out + residual
        return out


class WideResNet(nn.Module):
    """
    CIFAR-style WideResNet.

    depth should satisfy:
        depth = 6n + 4

    Examples:
        WRN-16-4: depth=16, widen_factor=4
        WRN-28-4: depth=28, widen_factor=4
        WRN-28-10: depth=28, widen_factor=10
    """

    def __init__(self, depth=28, widen_factor=6, drop_rate=0.2, num_classes=10):
        super().__init__()

        assert (depth - 4) % 6 == 0, "depth should be 6n + 4"
        n = (depth - 4) // 6

        channels = [
            16,
            16 * widen_factor,
            32 * widen_factor,
            64 * widen_factor,
        ]

        self.conv1 = nn.Conv2d(
            3,
            channels[0],
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

        self.stage1 = self._make_stage(
            in_channels=channels[0],
            out_channels=channels[1],
            num_blocks=n,
            stride=1,
            drop_rate=drop_rate,
        )

        self.stage2 = self._make_stage(
            in_channels=channels[1],
            out_channels=channels[2],
            num_blocks=n,
            stride=2,
            drop_rate=drop_rate,
        )

        self.stage3 = self._make_stage(
            in_channels=channels[2],
            out_channels=channels[3],
            num_blocks=n,
            stride=2,
            drop_rate=drop_rate,
        )

        self.bn = nn.BatchNorm2d(channels[3])
        self.relu = nn.ReLU(inplace=True)

        self._init_weights()

    def _make_stage(self, in_channels, out_channels, num_blocks, stride, drop_rate):
        layers = []

        layers.append(
            WideBasicBlock(
                in_channels=in_channels,
                out_channels=out_channels,
                stride=stride,
                drop_rate=drop_rate,
            )
        )

        for _ in range(1, num_blocks):
            layers.append(
                WideBasicBlock(
                    in_channels=out_channels,
                    out_channels=out_channels,
                    stride=1,
                    drop_rate=drop_rate,
                )
            )

        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(
                    m.weight,
                    mode="fan_out",
                    nonlinearity="relu",
                )
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.conv1(x)  # [B, 16, 32, 32]

        x = self.stage1(x)  # [B, 16*k, 32, 32]
        x = self.stage2(x)  # [B, 32*k, 16, 16]
        x = self.stage3(x)  # [B, 64*k, 8, 8]

        x = self.bn(x)
        x = self.relu(x)

        return x

In [6]:
class SVHN_CNN_multihead(nn.Module):

    def __init__(self, feature_dim=256, max_digits=5):
        self.max_digits = max_digits
        super().__init__()
        self.cnn = WideResNet(
            depth=28,
            widen_factor=4,
            drop_rate=0.2,
            num_classes=128,
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.length_head = nn.Linear(feature_dim, max_digits)

        # 用于 digit prediction：保留 5 个横向 slot
        self.slot_pool = nn.AdaptiveAvgPool2d((1, max_digits))
        self.digit_heads = nn.ModuleList([nn.Linear(feature_dim, 11) for _ in range(max_digits)])

    def forward(self, x):
        # feature_map: [B, C, H, W]
        feature_map = self.cnn(x)

        # ---------- length head ----------
        # [B, C, H, W] -> [B, C, 1, 1] -> [B, C]
        global_feat = self.global_pool(feature_map)
        global_feat = global_feat.flatten(1)

        length_logits = self.length_head(global_feat)  # [B, 5]

        # ---------- digit heads ----------
        # [B, C, H, W] -> [B, C, 1, 5]
        slot_feat = self.slot_pool(feature_map)

        # [B, C, 1, 5] -> [B, C, 5]
        slot_feat = slot_feat.squeeze(2)

        # [B, C, 5] -> [B, 5, C]
        slot_feat = slot_feat.permute(0, 2, 1)

        digit_logits = []
        for i in range(self.max_digits):
            # slot_feat[:, i, :] 是第 i 个横向 slot 的 feature
            digit_logits.append(self.digit_heads[i](slot_feat[:, i, :]))

        # list of 5 × [B, 11] -> [B, 5, 11]
        digit_logits = torch.stack(digit_logits, dim=1)

        return length_logits, digit_logits


model=SVHN_CNN_multihead().to(device)
# print(model)
num_epochs=100
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4, nesterov=True
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-4)
# print(model)

In [7]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_seq_correct = 0
    total_len_correct = 0
    total_digit_correct = 0
    total_digits = 0
    total_samples = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device).long()

        true_lengths = y[:, 0]  # [B], 1-5
        length_targets = true_lengths - 1
        digit_targets = y[:, 1:]  # [B, 5]

        length_logits, digit_logits = model(x)
        # length_logits: [B, 5]
        # digit_logits:  [B, 5, 11]

        loss_length = criterion(length_logits, length_targets)

        loss_digits = 0.0
        max_digits = digit_targets.size(1)

        for i in range(max_digits):
            loss_digits += criterion(digit_logits[:, i, :], digit_targets[:, i])

        loss = (loss_length + loss_digits) / (max_digits + 1)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        pred_lengths = length_logits.argmax(dim=1) + 1
        pred_digits = digit_logits.argmax(dim=2)

        total_len_correct += (pred_lengths == true_lengths).sum().item()

        pos = torch.arange(max_digits, device=device).unsqueeze(0)
        digit_mask = pos < true_lengths.unsqueeze(1)

        total_digit_correct += ((pred_digits == digit_targets) & digit_mask).sum().item()

        total_digits += digit_mask.sum().item()

        digits_all_correct = ((pred_digits == digit_targets) | ~digit_mask).all(dim=1)

        seq_correct = (pred_lengths == true_lengths) & digits_all_correct
        total_seq_correct += seq_correct.sum().item()

    avg_loss = total_loss / total_samples
    seq_acc = total_seq_correct / total_samples
    len_acc = total_len_correct / total_samples
    digit_acc = total_digit_correct / total_digits

    return avg_loss, seq_acc, len_acc, digit_acc


def evaluate(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_seq_correct = 0
    total_len_correct = 0
    total_digit_correct = 0
    total_digits = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            y = y.to(device).long()

            true_lengths = y[:, 0]  # [B], 1-5
            length_targets = true_lengths - 1
            digit_targets = y[:, 1:]  # [B, 5]

            length_logits, digit_logits = model(x)
            # length_logits: [B, 5]
            # digit_logits:  [B, 5, 11]

            loss_length = criterion(length_logits, length_targets)

            loss_digits = 0.0
            max_digits = digit_targets.size(1)

            for i in range(max_digits):
                loss_digits += criterion(digit_logits[:, i, :], digit_targets[:, i])

            loss = (loss_length + loss_digits) / (max_digits + 1)

            batch_size = x.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            # ---------- metrics ----------
            pred_lengths = length_logits.argmax(dim=1) + 1  # [B], 1-5
            pred_digits = digit_logits.argmax(dim=2)  # [B, 5]

            total_len_correct += (pred_lengths == true_lengths).sum().item()

            # 只统计真实存在的数字，不统计 padding
            pos = torch.arange(max_digits, device=device).unsqueeze(0)  # [1, 5]
            digit_mask = pos < true_lengths.unsqueeze(1)  # [B, 5]

            total_digit_correct += ((pred_digits == digit_targets) & digit_mask).sum().item()

            total_digits += digit_mask.sum().item()

            # sequence accuracy: 长度正确，并且所有真实数字都正确
            digits_all_correct = ((pred_digits == digit_targets) | ~digit_mask).all(dim=1)

            seq_correct = (pred_lengths == true_lengths) & digits_all_correct
            total_seq_correct += seq_correct.sum().item()

    avg_loss = total_loss / total_samples
    seq_acc = total_seq_correct / total_samples
    len_acc = total_len_correct / total_samples
    digit_acc = total_digit_correct / total_digits

    return avg_loss, seq_acc, len_acc, digit_acc

In [ ]:
from datetime import datetime

train_losses = []
train_seq_accs = []
train_len_accs = []
train_digit_accs = []

test_losses = []
test_seq_accs = []
test_len_accs = []
test_digit_accs = []

best_test_seq_acc = 0.0
epoch = 0

for _ in range(num_epochs):
    train_loss, train_seq_acc, train_len_acc, train_digit_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    test_loss, test_seq_acc, test_len_acc, test_digit_acc = evaluate(
        model, test_loader, criterion, device
    )

    train_losses.append(train_loss)
    train_seq_accs.append(train_seq_acc)
    train_len_accs.append(train_len_acc)
    train_digit_accs.append(train_digit_acc)

    test_losses.append(test_loss)
    test_seq_accs.append(test_seq_acc)
    test_len_accs.append(test_len_acc)
    test_digit_accs.append(test_digit_acc)

    scheduler.step()

    # 对 full SVHN，最重要的是 sequence accuracy
    if test_seq_acc > best_test_seq_acc:
        best_test_seq_acc = test_seq_acc
        torch.save(model.state_dict(), "SVHN_full_multihead.pt")

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Seq Acc: {train_seq_acc:.4f}, "
        f"Train Len Acc: {train_len_acc:.4f}, "
        f"Train Digit Acc: {train_digit_acc:.4f} | "
        f"Test Loss: {test_loss:.4f}, "
        f"Test Seq Acc: {test_seq_acc:.4f}, "
        f"Test Len Acc: {test_len_acc:.4f}, "
        f"Test Digit Acc: {test_digit_acc:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    epoch += 1

In [ ]:
import json

history = {
    "train_losses": train_losses,
    "train_seq_accs": train_seq_accs,
    "train_len_accs": train_len_accs,
    "train_digit_accs": train_digit_accs,
    "test_losses": test_losses,
    "test_seq_accs": test_seq_accs,
    "test_len_accs": test_len_accs,
    "test_digit_accs": test_digit_accs,
    "best_test_seq_acc": best_test_seq_acc,
}

with open("history.json", "w") as f:
    json.dump(history, f, indent=4)

print(f"History saved to: history.json")

In [ ]:
import matplotlib.pyplot as plt
save_dir = "./svhn_full_results"
os.makedirs(save_dir, exist_ok=True)
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, test_losses, label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("SVHN Full Multi-head Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_fig_path = os.path.join(save_dir, "SVHN_full_loss.png")
plt.savefig(loss_fig_path, dpi=300)
plt.show()

print(f"Loss figure saved to: {loss_fig_path}")
# 0.9643

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_seq_accs, label="Train Seq Acc")
plt.plot(epochs, test_seq_accs, label="Test Seq Acc")

plt.xlabel("Epoch")
plt.ylabel("Sequence Accuracy")
plt.title("SVHN Full Multi-head Sequence Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()

seq_acc_fig_path = os.path.join(save_dir, "SVHN_full_sequence_accuracy.png")
plt.savefig(seq_acc_fig_path, dpi=300)
plt.show()

print(f"Sequence accuracy figure saved to: {seq_acc_fig_path}")

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_len_accs, label="Train Length Acc")
plt.plot(epochs, test_len_accs, label="Test Length Acc")

plt.xlabel("Epoch")
plt.ylabel("Length Accuracy")
plt.title("SVHN Full Multi-head Length Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()

len_acc_fig_path = os.path.join(save_dir, "SVHN_full_length_accuracy.png")
plt.savefig(len_acc_fig_path, dpi=300)
plt.show()

print(f"Length accuracy figure saved to: {len_acc_fig_path}")


plt.figure(figsize=(8, 5))
plt.plot(epochs, train_digit_accs, label="Train Digit Acc")
plt.plot(epochs, test_digit_accs, label="Test Digit Acc")

plt.xlabel("Epoch")
plt.ylabel("Digit Accuracy")
plt.title("SVHN Full Multi-head Digit Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()

digit_acc_fig_path = os.path.join(save_dir, "SVHN_full_digit_accuracy.png")
plt.savefig(digit_acc_fig_path, dpi=300)
plt.show()

print(f"Digit accuracy figure saved to: {digit_acc_fig_path}")